# Route objective variants — checking a candidate formula against the shipped baselines

**§0 builds the one thing this needs that doesn't exist yet**: raw top-10 rankings per route (`route_rankings`), so a new objective can be re-scored without re-running retrieval every time. `labels.parquet` (the golden set) already has everything for the *shipped* objective — it stores each route's final blended score — but not the ranking that produced it, so a different formula (this notebook's Recall term especially) can't be reconstructed from it. That's a gap in what gets persisted, not a problem with the golden set itself, and nothing here reads or writes `labels.parquet`.

**§1 onward is zero retrieval cost**: once §0 (or `route_labels.ipynb` §17, extended the same way) has written a lane's `{lane}_oracle/rows.parquet`, every later run of this notebook just reads it back — no Qdrant involved.

**What's being checked.** The shipped objective is `0.7·HitRate@1 + 0.3·NDCG@10`. The candidate adds a third term:

```
0.7·HitRate@1 + 0.3·NDCG@10² - c·Recall@10·(1 - NDCG@10)
```

`Recall@10·(1-NDCG@10)` is the share of judged-relevant docs that made it into the top-10 but were **not** converted into a good ranking — the RRF dilution mode `fusion.py`'s `PureRRFStrategy` already documents (SPEC d37g): a confident top-1 from one route can be buried by fusion while still counting toward recall. It's multiplicative in Recall, not a flat subtraction, so a route with little recall to begin with isn't punished twice for also having low NDCG — that case already scores low from the positive terms alone.

Squaring NDCG stretches the mid-range (0.6 → 0.36, 0.8 → 0.64) where ties cluster, which is the shipped objective's actual `all_tied` problem.

**Scope.** `Recall@10` here is bounded by the same top-10 window as `NDCG@10` — `route_rankings` only ever stores each route's top 10 ids — so this measures within-window rank dilution, not corpus-wide recall. The objective is defined locally in this notebook, not in `objective.py`: it hasn't earned a place in shipped code yet, this notebook is how it earns one.

## 0 — Build the oracle cache for every lane

**Path note:** the package's own default (`labels.py`'s `DEFAULT_OUT_DIR`, computed as `Path(__file__).parent.parent / "data"`) resolves to `src/data`, not a repo-root `data/` — this notebook lives in `notebooks/`, so every path below is `../src/data`, not `data`. `labels.parquet` and every lane's snapshot already live there.

If every lane's `{lane}_oracle/rows.parquet` already exists under `src/data/route_labels/` (it should — that's the same place `labels.parquet` lives), `build_or_load` finds them immediately and this cell finishes in seconds, no retrieval. It only actually queries Qdrant for a lane whose cache is genuinely missing.

Read-only against Qdrant — only `query_points` search calls, no upserts, no writes to `labels.parquet`. One bad lane is reported and skipped, not fatal to the sweep — same resilience contract as `route_labels.ipynb` §17.

In [ ]:
import os

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance

load_dotenv(".env") or load_dotenv("../.env")

DENSE_MODEL, DENSE_SIZE = "BAAI/bge-small-en-v1.5", 384
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY"),
)
print("qdrant collections:", [c.name for c in client.get_collections().collections])

In [ ]:
import pandas as pd

from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy,
    PureRRFStrategy,
    SparseOnlyStrategy,
)
from hybrid_search_rrf_dataset.golden import GoldenRoutingBuilder
from hybrid_search_rrf_dataset.indexer import EmbeddingConfig
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.lanes import LANES
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.retrieval import QuerySubset, SnapshotDataset

# the two anchor collections predate the {source.name}_routes convention
# (route_labels.ipynb §4, §12); override so this reuses their indexes.
COLLECTION_OVERRIDE = {
    "beir-nfcorpus": "nfcorpus_routes",
    "msmarco-passage-dev": "msmarco_routes",
}


def source_name(key: str) -> str:
    return LANES[key].source.name


def collection_of(key: str) -> str:
    return COLLECTION_OVERRIDE.get(key, f"{source_name(key)}_routes")


# the query universe comes straight from labels.parquet, NOT CellFill().build():
# CellFill produces a curated 44-cell-archetype subset for a different job
# (checking each cell's predicted route against the measured one) and is
# capped, by its own quota design, far below the golden set's real size --
# it was never the right source for "every query already labelled."
labels_on_disk = RouteLabels(pd.DataFrame(columns=["dataset", "query_id"])).load()
selection = labels_on_disk[["dataset", "query_id"]].drop_duplicates()
labels = RouteLabels(selection)

dense_cfg = EmbeddingConfig(
    name="dense_base", model_id=DENSE_MODEL, kind="dense",
    size=DENSE_SIZE, distance=Distance.COSINE,
)
sparse_cfg = EmbeddingConfig(name="sparse_base", model_id=SPARSE_MODEL, kind="sparse")

In [ ]:
DATA_DIR = "../src/data"  # see the path note above

for key in LANES:
    try:
        src = SnapshotDataset(source_name(key), path=DATA_DIR)
        subset = QuerySubset(src, labels.rows_for(key)["query_id"])
        strat_args = (client, collection_of(key), dense_cfg, sparse_cfg)
        lane_dense, lane_hybrid, lane_sparse = (
            DenseOnlyStrategy(*strat_args),
            PureRRFStrategy(*strat_args),
            SparseOnlyStrategy(*strat_args),
        )
        rows = GoldenRoutingBuilder(
            lane_dense, lane_hybrid, lane_sparse,
            objective=RouterObjective(min_relevance=LANES[key].min_relevance),
        ).build_or_load(subset, path=f"{DATA_DIR}/route_labels/{key}_oracle")
        print(f"{key:38s} cached {len(rows):>6,} rows")
    except Exception as error:
        print(f"{key:38s} SKIPPED: {error}")

## 1 — Load every cached lane oracle

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import Markdown, display


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))

In [ ]:
from pathlib import Path

from hybrid_search_rrf_dataset.golden import GoldenRoutingBuilder
from hybrid_search_rrf_dataset.lanes import LANES
from hybrid_search_rrf_dataset.qrels import QrelStore
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

DATA_DIR = "../src/data"  # see §0's path note — package default is src/data, not data/


def load_lane_oracle(key: str):
    path = Path(DATA_DIR) / "route_labels" / f"{key}_oracle"
    if not (path / "rows.parquet").exists():
        return None
    return GoldenRoutingBuilder.load(path=path)


lane_rows, lane_lookup, skipped = {}, {}, []
for key in LANES:
    rows = load_lane_oracle(key)
    if rows is None:
        skipped.append(key)
        print(f"{key:40s} skip — not cached")
        continue
    lane_source = SnapshotDataset(LANES[key].source.name, path=DATA_DIR)
    lane_rows[key] = rows
    lane_lookup[key] = QrelStore.from_dataset(lane_source).lookup(lane_source.name)
    print(f"{key:40s} loaded {len(rows):>6,} rows")


all_rows = [(key, row) for key, rows in lane_rows.items() for row in rows]
print(f"pooled oracle rows: {len(all_rows):,} across {len(lane_rows)} of {len(LANES)} lanes")
if skipped:
    print(f"not yet cached ({len(skipped)}): {', '.join(skipped)}")

## 2 — Candidate objective: penalize wasted recall

`hit_weight`/`ndcg_weight` default to the shipped 0.7/0.3 split; `waste_weight` (`c`) is left unpinned and swept below instead of hand-picked, per the project's rule that thresholds get read off real data, not typed in.

In [ ]:
from pydantic import Field

from hybrid_search_rrf_dataset.objective import Objective


class WastedRecallObjective(Objective):
    """`hit_weight`·HitRate@1 + `ndcg_weight`·NDCG@k² - `waste_weight`·Recall@k·(1-NDCG@k).

    The penalty is proportional to Recall, so it only bites when there was
    real recall to waste — a route with low Recall AND low NDCG is already
    scored low by the positive terms and isn't double-punished here.
    """

    hit_weight: float = Field(default=0.7, ge=0.0)
    ndcg_weight: float = Field(default=0.3, ge=0.0)
    waste_weight: float = Field(default=0.3, ge=0.0)

    @property
    def name(self) -> str:
        return (
            f"{self.hit_weight:g}*HR@1+{self.ndcg_weight:g}*NDCG@{self.top_k}^2"
            f"-{self.waste_weight:g}*wasted_recall"
        )

    @property
    def decisive_margin(self) -> float:
        # unlike RouterObjective, the penalty can pull a rank-1 hit below
        # ndcg_weight, so no fixed gap certifies a top-1 separation.
        return float("inf")

    def assess(
        self, ranking: dict[str, float], gold_qrel: dict[str, int]
    ) -> tuple[float, list[str]]:
        top = self.ordered(ranking)
        relevant = self.relevant(gold_qrel)
        if not relevant:
            return 0.0, top
        hit = self.hit_weight if top and top[0] in relevant else 0.0
        ndcg = self.ndcg(ranking, relevant)
        recall = len(set(top) & relevant.keys()) / len(relevant)
        score = (
            hit
            + self.ndcg_weight * ndcg**2
            - self.waste_weight * recall * (1 - ndcg)
        )
        return score, top

## 3 — Relabel against every baseline, `all_tied` included

Same `relabel` shape as `route_labels.ipynb` §9, extended to also classify each row's `outcome_shape` (reusing `labels.py`'s tested classifier rather than re-deriving tie detection) so `all_tied` can be read per objective, not just the pick distribution and flip rate.

In [ ]:
from tqdm.auto import tqdm

from hybrid_search_rrf_dataset.fusion import derive_route
from hybrid_search_rrf_dataset.labels import ALL_TIED, outcome_shape
from hybrid_search_rrf_dataset.objective import NDCGObjective, RouterObjective


def relabel(make_objective, desc: str) -> pd.DataFrame:
    # derive_route, not argmax: ties resolve to the cheapest route (SPEC d41),
    # matching exactly how labels.parquet's `route` column is derived — a plain
    # max() here would silently default every tie to dense_only (StrategyName's
    # enum order), corrupting the flip-rate comparison against `shipped`.
    picks, shapes = [], []
    for key, row in tqdm(all_rows, desc=desc):
        objective = make_objective(LANES[key].min_relevance)
        gold = lane_lookup[key].get(row.query_id, {})
        scored = {
            route: objective.assess(
                {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
            )[0]
            for route, ids in row.route_rankings.items()
        }
        picks.append(str(derive_route(scored)))
        shapes.append(outcome_shape(scored))
    return pd.DataFrame({"pick": picks, "shape": shapes})


shipped = pd.Series([str(row.strategy_name) for _, row in all_rows])
variants = {
    "1·HR@1": lambda mr: RouterObjective(hit_weight=1, ndcg_weight=0, min_relevance=mr),
    "0.7·HR@1 + 0.3·NDCG@10  (shipped)": lambda mr: RouterObjective(min_relevance=mr),
    "0.5·HR@1 + 0.5·NDCG@10": lambda mr: RouterObjective(hit_weight=0.5, ndcg_weight=0.5, min_relevance=mr),
    "bare NDCG@10 (no top-1 term)": lambda mr: NDCGObjective(min_relevance=mr),
    "shipped, stricter min_relevance+1": lambda mr: RouterObjective(min_relevance=mr + 1),
    **{
        f"0.7·HR@1 + 0.3·NDCG@10² - {c:g}·wasted_recall": (
            lambda mr, c=c: WastedRecallObjective(waste_weight=c, min_relevance=mr)
        )
        for c in (0.15, 0.3, 0.5, 1.0)
    },
}

rows_out = []
for name, make_objective in variants.items():
    out = relabel(make_objective, desc=name)
    result = {
        "objective": name,
        "flips": int((out["pick"] != shipped).sum()),
        "flip rate": f"{(out['pick'] != shipped).mean() * 100:.1f}%",
        **{f"picks {r}": int((out["pick"] == r).sum())
           for r in ("dense_only", "pure_rrf", "sparse_only")},
        "all_tied": int((out["shape"] == ALL_TIED).sum()),
    }
    print(f"  -> {result['flips']:,} flips ({result['flip rate']}), {result['all_tied']:,} all_tied")
    rows_out.append(result)
show(pd.DataFrame(rows_out))

## Reading the table

- **`flips`/`flip rate`** — rows where this objective's pick differs from the shipped label. High flips + lower `all_tied` is the interesting quadrant: the candidate is actually resolving ties, not just relabelling noise.
- **`all_tied`** — should move as `c` (`waste_weight`) grows, if the penalty is doing anything: it can only break a tie between two routes when their Recall/NDCG mix differs, which is exactly the case a flat `0.7·HR + 0.3·NDCG` blend can't see once both routes already agree on HitRate@1.
- **If `all_tied` barely moves even at `c=1.0`**, the wasted-recall term isn't the lever for this dataset's ties — worth checking whether ties are actually driven by identical top-1 docs across routes (a structural tie no score reshaping can break) rather than by this Recall/NDCG gap.

## 4 — Are `all_tied` rows structurally tied, or just formula-tied?

The premise behind §3's `wasted_recall` sweep barely moving `all_tied`: for every row the **shipped** objective calls a tie, do all three routes return the *exact same ranked list* (`route_rankings["dense_only"] == route_rankings["pure_rrf"] == route_rankings["sparse_only"]`)? If so, every per-query positional metric — HitRate, NDCG, Recall, MRR, any reweighting of them — computes the identical score for all three routes, and the tie is a floor no formula search can lower (SPEC d41: serve the cheapest, that's correct behavior here, not a bug). If the rankings differ but still score equal, the tie is an artifact of *this* formula's arithmetic, and a different one could in principle break it.

In [10]:
shipped_shapes = []
for key, row in all_rows:
    objective = RouterObjective(min_relevance=LANES[key].min_relevance)
    gold = lane_lookup[key].get(row.query_id, {})
    scored = {
        route: objective.assess(
            {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
        )[0]
        for route, ids in row.route_rankings.items()
    }
    shipped_shapes.append(outcome_shape(scored))

tied_rows = [
    row for (key, row), shape in zip(all_rows, shipped_shapes) if shape == ALL_TIED
]
identical = sum(
    row.route_rankings["dense_only"]
    == row.route_rankings["pure_rrf"]
    == row.route_rankings["sparse_only"]
    for row in tied_rows
)

print(f"{len(tied_rows):,} all_tied rows under the shipped objective")
print(
    f"  {identical:,} ({identical / len(tied_rows) * 100:.1f}%) have "
    "identical top-10 rankings across all three routes -- structurally "
    "tied, no formula can break these"
)
print(
    f"  {len(tied_rows) - identical:,} "
    f"({(len(tied_rows) - identical) / len(tied_rows) * 100:.1f}%) have "
    "differing rankings that just happen to score equal under the shipped "
    "objective -- these ARE, in principle, formula-breakable"
)

15,015 all_tied rows under the shipped objective
  0 (0.0%) have identical top-10 rankings across all three routes -- structurally tied, no formula can break these
  15,015 (100.0%) have differing rankings that just happen to score equal under the shipped objective -- these ARE, in principle, formula-breakable


## 5 — Confirm the mechanism: single relevant doc, same rank

§4 showed 0% of `all_tied` rows have identical top-10 *lists* but 100% score identically anyway. NDCG/HR/Recall are pure functions of *where the relevant docs sit*, blind to which irrelevant docs fill the rest — so the predicted mechanism is: most of these queries have exactly one relevant doc, and all three routes happen to rank it at the same position. This checks both parts directly: the relevant-doc-count distribution, and whether the relevant-doc rank *positions* (not the full lists) match across routes — the generalization that also covers any multi-relevant-doc ties.

In [11]:
tied_with_key = [
    (key, row)
    for (key, row), shape in zip(all_rows, shipped_shapes)
    if shape == ALL_TIED
]

relevant_counts, position_matches = [], []
for key, row in tied_with_key:
    gold = lane_lookup[key].get(row.query_id, {})
    relevant = sorted(d for d, r in gold.items() if r >= LANES[key].min_relevance)
    relevant_counts.append(len(relevant))
    positions = {
        route: tuple(ids.index(d) if d in ids else None for d in relevant)
        for route, ids in row.route_rankings.items()
    }
    position_matches.append(len(set(positions.values())) == 1)

counts = pd.Series(relevant_counts).value_counts().sort_index()
print("relevant-doc count among all_tied rows:")
print(counts.head(10))
one_doc_share = (pd.Series(relevant_counts) == 1).mean() * 100
print(f"-> {one_doc_share:.1f}% of all_tied rows have exactly 1 relevant doc")
print()

match_rate = pd.Series(position_matches).mean() * 100
print(
    f"{match_rate:.1f}% of all_tied rows have identical relevant-doc rank "
    "positions across all three routes (the exact condition that forces "
    "NDCG/HR/Recall to tie, regardless of how many relevant docs there are)"
)

relevant-doc count among all_tied rows:
1     14025
2       628
3       181
4        86
5        33
6        20
7        12
8         6
9         6
10        4
Name: count, dtype: int64
-> 93.4% of all_tied rows have exactly 1 relevant doc

96.6% of all_tied rows have identical relevant-doc rank positions across all three routes (the exact condition that forces NDCG/HR/Recall to tie, regardless of how many relevant docs there are)


## 6 — Raw-score margin on `all_tied` rows: dense vs. sparse (BM25)

§4/§5 showed HR/NDCG/Recall can't see anything beyond relevant-doc rank, so they're structurally blind on ~96.6% of `all_tied` rows. `route_raw_scores` (added this session) carries what they can't: each route's own retrieval score, in `route_rankings` order. This looks at each route's own top1-vs-top2 margin on exactly the tied rows, to see whether there's a usable confidence signal in there at all before designing an objective around it.

`sparse_only` **is** BM25 here (`SPARSE_MODEL = "Qdrant/bm25"` in `fusion.py`) — one route, not two. The margin is reported both raw and relative (`(top1-top2)/top1`); raw margins aren't comparable across routes (cosine similarity, BM25, and RRF's fused score are different scales), the relative version is the closest thing to an apples-to-apples read.

In [12]:
def margin(row, route):
    raw = row.route_raw_scores.get(route, [])
    if len(raw) < 2:
        return None, None
    top1, top2 = raw[0], raw[1]
    abs_margin = top1 - top2
    rel_margin = (abs_margin / top1) if top1 else None
    return abs_margin, rel_margin


records = []
for key, row in tied_with_key:
    entry = {"dataset": key, "query_id": row.query_id}
    for route in ("dense_only", "sparse_only", "pure_rrf"):
        abs_m, rel_m = margin(row, route)
        entry[f"{route}_abs"] = abs_m
        entry[f"{route}_rel"] = rel_m
    records.append(entry)

margins_df = pd.DataFrame(records)
print(f"{len(margins_df):,} all_tied rows with raw scores")
show(margins_df[[c for c in margins_df.columns if c.endswith("_rel")]].describe().round(4))

paired = margins_df.dropna(subset=["dense_only_rel", "sparse_only_rel"])
sparse_bigger = (paired["sparse_only_rel"] > paired["dense_only_rel"]).mean() * 100
print(
    f"of {len(paired):,} tied rows with both margins: sparse's relative "
    f"margin is bigger than dense's in {sparse_bigger:.1f}% of them"
)

15,015 all_tied rows with raw scores


|   dense_only_rel |   sparse_only_rel |   pure_rrf_rel |
|-----------------:|------------------:|---------------:|
|       15015      |        14931      |     15015      |
|           0.112  |            0.2831 |         0.5032 |
|           0.0753 |            0.1966 |         0.1523 |
|           0      |            0      |         0      |
|           0.0478 |            0.1096 |         0.4167 |
|           0.1045 |            0.2656 |         0.5417 |
|           0.1661 |            0.4295 |         0.6381 |
|           0.3952 |            0.9239 |         0.6667 |

of 14,931 tied rows with both margins: sparse's relative margin is bigger than dense's in 82.9% of them
